# Tutorial 0 — Start here

Estimated time: 10 minutes. No simulation, no model runs — this is orientation.

This repository reproduces and extends **Neve-Oz, Sherman & Raveh, "Bayesian
metamodeling of early T-cell antigen receptor signaling accounts for its nanoscale
activation patterns"** (*Frontiers in Immunology*, 2024).

There are two ways to read it, and they answer different questions. This notebook helps
you pick, and checks that the one you pick will actually run on your machine.

> **Citation note.** `README.md` gives the DOI as `10.3389/fimmu.2024.1437672`; the
> 2026-03-08 entry in `Status.md` gives `10.3389/fimmu.2024.1412221` for the same paper.
> One is wrong; both are recorded here rather than silently choosing.

## The two tracks

**Track A — the model track.** How does one biophysical model work? Currently this
means kinetic segregation: the biology, the energy function, the Monte Carlo scheme,
what each parameter does, and how to measure the result without fooling yourself. Runs
on numpy + a C binary. **No framework needed.**

**Track B — the metamodel track.** How do four separately-built models get combined
into one joint posterior? Sweeps, surrogates, couplings, and the paper's figures.
Drives the `bayesmm` CLI. **Needs the `bayesian-metamodeling` framework**, and PyMC for
the surrogate and inference steps.

| | Track A — model | Track B — metamodel |
|---|---|---|
| Location | `models/kinetic_segregation/KS_1` … `KS_5` | `01_explore_models` … `04_reproduce_figures` |
| Question | *why does this model behave like this?* | *how do models constrain each other?* |
| Needs | numpy, matplotlib, a C compiler | `bayesmm`; PyMC for `02`/`03` |
| Runtime | seconds per notebook | minutes (sampling) |
| Executed in CI | yes | no (framework lives in the parent repo) |

## Which should you read?

| If you want to… | Go to |
|---|---|
| Understand what kinetic segregation *is* | **KS 1** |
| See the energy terms and the MC algorithm | **KS 2** |
| Run the simulator and read its output | **KS 3** |
| Know what κ, CD45 height, or binding mode do | **KS 4** |
| Choose a depletion metric, or avoid the CLI traps | **KS 5** |
| Run all four partial models once | `01_explore_models` |
| Fit surrogates to sweep data | `02_fit_surrogates` |
| Sample the coupled joint posterior | `03_metamodel_inference` |
| Reproduce the pTCR ring figure | `04_reproduce_figures` |

**If you are new, read KS 1 first even if you came for the metamodel.** Track B treats
each partial model as a box that emits a number; Track A is where you learn what that
number means and how badly it can mislead you.

## What the four partial models are

| # | Model | Question | Contributes |
|---|---|---|---|
| 1 | Membrane topography | Where are the membranes close enough to count as contact? | contact geometry |
| 2 | **Kinetic segregation** | Where does CD45 end up, given that geometry? | `depletion_width_nm` |
| 3 | Lck activity | How far from the CD45 boundary does active kinase persist? | `mean_lck_activity` |
| 4 | TCR phosphorylation | Where do phosphorylated ITAMs accumulate? | pTCR profile |

The scientific payoff is a **ring**: CD45 is most depleted at the centre of the contact,
but active Lck decays over ~70 nm from the boundary, so the product of "kinase present"
and "phosphatase absent" peaks in an annulus rather than at the centre.

They are **coupled, not chained** — the metamodel constrains them jointly rather than
piping each output forward as a fixed input, so evidence about one sharpens the others.

## Will it run here?

The check below is honest about which track is available to you. Neither answer is
wrong — Track A deliberately avoids the framework so it works in a bare checkout.

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path


def find_repo_root(start=None):
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "models" / "kinetic_segregation" / "CMakeLists.txt").is_file():
            return cand
    raise RuntimeError(f"could not locate the tcr_signaling repo above {here}")


ROOT = find_repo_root()
KS_DIR = ROOT / "models" / "kinetic_segregation"


def have_module(name):
    try:
        __import__(name)
        return True
    except Exception:
        return False


binary = (KS_DIR / "ks_gpu").exists()
compiler = shutil.which("cmake") is not None and (
    shutil.which("c++") is not None or shutil.which("clang++") is not None
)
bayesmm = subprocess.run(["bayesmm", "--version"],
                         capture_output=True, text=True).returncode == 0

checks = [
    ("python", True, sys.version.split()[0]),
    ("numpy", have_module("numpy"), "Track A"),
    ("matplotlib", have_module("matplotlib"), "Track A"),
    ("cmake + C++ compiler", compiler, "Track A (builds the model)"),
    ("ks_gpu binary built", binary, "Track A (or run `make` in models/kinetic_segregation)"),
    ("bayesmm", bayesmm, "Track B"),
    ("pymc", have_module("pymc"), "Track B: notebooks 02, 03"),
    ("sbi", have_module("sbi"), "Track B: optional sbi_npe backend"),
]
width = max(len(n) for n, _, _ in checks)
for name, ok, note in checks:
    print(f"  {'OK ' if ok else '-- '} {name:<{width}}   {note}")

track_a = all([have_module("numpy"), have_module("matplotlib"), compiler or binary])
track_b = bayesmm
print()
print(f"Track A (kinetic segregation) : {'ready' if track_a else 'not ready'}")
print(f"Track B (metamodel)           : {'ready' if track_b else 'not ready -- install bayesian-metamodeling'}")
if track_a and not track_b:
    print("\n-> Start at models/kinetic_segregation/KS_1_Kinetic_Segregation.ipynb")

  OK  python                 3.14.6
  OK  numpy                  Track A
  OK  matplotlib             Track A
  OK  cmake + C++ compiler   Track A (builds the model)
  OK  ks_gpu binary built    Track A (or run `make` in models/kinetic_segregation)
  OK  bayesmm                Track B
  --  pymc                   Track B: notebooks 02, 03
  --  sbi                    Track B: optional sbi_npe backend

Track A (kinetic segregation) : ready
Track B (metamodel)           : ready


## Vocabulary

- **Partial model** — one of the four biophysical models, independently parameterised
  and independently validated.
- **Sweep / DOE** — running a model over a planned set of input combinations.
- **Surrogate** — a fast *probabilistic* stand-in fit to a sweep. It predicts the
  model's output at unseen inputs *and reports its own uncertainty*; that second part is
  what makes it usable inside a Bayesian metamodel.
- **Coupling** — an assertion that two variables in different models are the same
  physical quantity, or related by a known transform. `gaussian_link` says "these should
  agree, to within σ"; `deterministic` says "these are equal by construction".
- **Joint posterior** — the distribution over all models' variables *after* the
  couplings have been imposed. Narrower than any model alone, because each constrains
  its neighbours.
- **Depletion width** — the KS model's headline observable: the radial gap between the
  CD45 and TCR distributions. KS 3 and KS 5 explain when it means what you think.

## Three things worth knowing before you start

1. **Grid resolution is not a speed/quality dial in the KS model.** If
   `dx = patch_size/grid_size` is much larger than `sigma_r` (2 nm), the pMHC influence
   weight becomes exactly zero and the TCR attraction switches off entirely. KS 3
   measures this. The checked-in specs currently sit in that regime.
2. **A metric reporting "no effect" may be saturated rather than insensitive.** KS 4
   shows `u_assoc` looking inert on one observable and clearly active on another.
3. **Two of the eight depletion metrics are `null` unless you pass
   `--monitor-binding`** — and they happen to be the two that survive an off-centre
   contact. KS 5 covers this.

In [2]:
assert ROOT.is_dir() and KS_DIR.is_dir()
assert (ROOT / "notebooks" / "models" / "kinetic_segregation").is_dir(), \
    "Track A notebooks are missing"
assert len(list((ROOT / "notebooks" / "models" / "kinetic_segregation").glob("KS_*.ipynb"))) >= 5
print("[Tutorial_0 self-check OK]")

[Tutorial_0 self-check OK]
